# Running The Alignment Process

This notebook contains a workflow for setting up the pre and post PUNKST Atlas alignment. It is intented to be run on one sample at time. This is not intented to be a full tutorial on how to use any of the specific software used in this workflow. It is simply the minimum information for completeing the task explained above. For more detailed explorations of the included software, tutorial and website links can be found below.

## Software Informtiaon:

- Qupath: 
    - Download: https://qupath.github.io/
    - Tutorials and further info: https://qupath.readthedocs.io/en/stable/docs/tutorials/index.html
- Abba: 
    - Download: https://abba-documentation.readthedocs.io/en/v0.11.0/installation/installation.html
        - This is also the reference page for ABBA. This will contain tutorials and general software information
        - If on Windows, the standalone installer is the easiest method if FIJI is not already installed.

- Conda: 
 

## Code Setup

In [2]:
import pandas as pd
import geopandas as gpd
import polars as pl
from pathlib import Path
#from shapely.geometry import Point

## Step 1 - Creating a Qupath Project

- Start Qupath Application
- Create Project
- Choose an existing folder to store all project files in, or make a new folder
- Name the project file
- Now back in the qupath menu, click add image
- Select Ch000 .tif file
- Ensure the image loads correctly
- File - Save

## Step 2 - Starting in ABBA
- Start the ABBA application
- Seach ABBA in the Fiji search bar and click ABBA Start
- click run
- import the abba project file that was just created ( with all default settings)
- now in the bottom right corner you should see a the name of the slice with colors, click each of the 4 boxes to bring the slice into view

## Step 3 - Aligning the Atlas

- NOTE: This program is very computationally heavy and can be prone to crashes, make sure work is saved often.
- Grab the box above the slice and drag it below the atlas slice that seems closest to the imported slice
- Register -> affline -> interactive transform
    - these setting allow you to get the relative rotation and sizing as close to the atlas slice as possible, it does not have to be perfect but getting it somewhat close will make things easier
- Register -> spline -> Elastix
    - This is an automatic registration step. Play around with matching the different channels and see how it warps.
- Register -> spline -> Big Warp
    - This is the manual registration tool
    - Meant for fine grain adjustments after a few Elastic runs have gotten the sample relatively close.



## Step 4

Export the project:
Export -> QuPath -> Export Qupath Registrations(toggle delete old ROIs)

## Step 5

Grab the GeoJSON files through Qupath:  
- Open the Qupath project file that was created at the start
- go to extensions -> ABBA -> load atlas annotations into open image
    - Make sure you clicked one of the channels files and the left and that there is an actual image on the screen,
        otherwise it will not work and you will get an error
    - Do not split -> change the naming property from ID to atlas_id
- if you can now see that registrations overlayed onto the brain image:
    - File -> Export Objects as GeoJSON -> check pretty JSON -> OK and save the file


# Crop regions of interest

In [ ]:
in_parq = r"S:\Anshutz\Cruz-Martin_Lab\projects\TBI_Project\data\20251013_FT_24hrs_Sag9_ID57476\Transcripts\transcripts.parquet"

pdf = pl.read_parquet(in_parq)


# Merge with Ficture output

Method 2: The Scripted Export (For Automation)
Since you are building a pipeline, you’ll want to automate this so you don't have to click through menus for every slice. Open the Script Editor in QuPath (Automate > Show Script Editor) and use this snippet:

Since you are automating this, you can use the geopandas library. It is built to handle this exact JSON format (GeoJSON) and perform "Spatial Joins."

Here is the code to map your Ficture transcripts to these specific atlas regions:

In [ ]:


# 1. Load the QuPath GeoJSON (Atlas boundaries)
# This converts the JSON you showed into a table of shapes
atlas_gdf = gpd.read_file("your_qupath_export.geojson")

# 2. Load your Ficture transcripts
# Assuming your Ficture file is a CSV/TSV with 'x' and 'y' columns
ficture_data = pd.read_csv("ficture_transcripts.csv")

# 3. Convert Ficture points into a GeoDataFrame
# IMPORTANT: Ensure 'x' and 'y' match the scale of the GeoJSON numbers
geometry = [Point(xy) for xy in zip(ficture_data['x'], ficture_data['y'])]
transcripts_gdf = gpd.GeoDataFrame(ficture_data, geometry=geometry)

# 4. Perform the "Point-in-Polygon" Join
# This checks which brain region polygon contains each transcript point
mapped_data = gpd.sjoin(transcripts_gdf, atlas_gdf, how="left", predicate="within")

# 5. Save the result
# Your new file will have the original Ficture data + a 'classification' column (the atlas region)
mapped_data.to_csv("ficture_with_atlas_regions.csv", index=False)